# 厨二病ベクトルの作成（all-MiniLM-L6-v2）

## 1. 文埋め込み（BERT）

In [3]:
!pip -q install -U sentence-transformers transformers

In [4]:
from pathlib import Path

def load_lines(path):
    lines = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

chuni = load_lines("/content/chuni_sentences.txt")
normal = load_lines("/content/normal_sentences.txt")

print("厨二病文数:", len(chuni))
print("普通文数:", len(normal))
print("厨二病サンプル:", chuni[0])
print("普通文サンプル:", normal[0])

厨二病文数: 130
普通文数: 130
厨二病サンプル: 妖精さんたち！私に力を貸して！！！魅惑の香りに飲み込まれなさい"ポイズン・ベリー"！！！
普通文サンプル: コノリジン(Conolidine)は、アルカロイドの1つである


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 日本語でも比較的安定して使える多言語SBERT
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

E_chu = model.encode(chuni, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
E_nor = model.encode(normal, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

mu_chu = E_chu.mean(axis=0)
mu_nor = E_nor.mean(axis=0)

v_chu = mu_chu - mu_nor
v_chu = v_chu / np.linalg.norm(v_chu)  # 正規化（コサイン用）

np.save("chuni_vector.npy", v_chu)

print("v_chu shape:", v_chu.shape)
print("chuni_vector.npy を保存しました")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'FetchError: Could not fetch resource at https://colab.research.google.com/userdata/get?authuser=0&notebookid=1S0b7jWyRbc8rw39ilFKAMcbU0Da8yxxC&key=HF_TOKEN: 401  '.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

v_chu shape: (384,)
chuni_vector.npy を保存しました


## 2. 厨二病ベクトルの定義

In [6]:
import numpy as np

# ベクトル読み込み
v_chu = np.load("chuni_vector.npy")

# すでに normalize_embeddings=True なので内積＝コサイン類似度
scores_chu = E_chu @ v_chu
scores_nor = E_nor @ v_chu

topk = 10

print("\n=== 厨二病文：上位10（厨二病度が高い） ===")
idx = np.argsort(-scores_chu)[:topk]
for i in idx:
    print(f"{scores_chu[i]:+.3f}  {chuni[i]}")

print("\n=== 普通文：下位10（厨二病度が低い） ===")
idx2 = np.argsort(scores_nor)[:topk]
for i in idx2:
    print(f"{scores_nor[i]:+.3f}  {normal[i]}")


=== 厨二病文：上位10（厨二病度が高い） ===
+0.439  祭壇に捧げられし、贄（にえ）よ。我が血肉となり力を与え給え！！...まてぇぇぇええい！？そこは我が領域で、我が育てた贄（にえ）だぞ！
+0.405  ふははは、計画通りだ！！！さっきのが全力だと思ってただろ！！違うぞさっきのはまだ60%しか力を出してないのさ。お前はもう終わりだ！
+0.399  よせ！その手を離すんだ！...それは開けてはいけない！その中には禁断の書が入っている。..っ？！だめだって！！！
+0.397  我が財をこの世界に譲渡する！！ぽちっ...召喚召喚召喚だぁぁぁぁああ！！！来い！来い！来ーーーい！....これで召喚は終わりか？...終わった。我が財が消え失せた。
+0.383  ふふふ、針怖いわけがないだろう。苦しゅうないぞ。早く終わらせるのだ。......。まて...それが注射か......思ったよりも太くないか...？？いやだぁああ！！！！
+0.383  うわっ！？なんだ！？叫びを止まるんだ！！封印されし、我のデバイスが暴走を始めたかッ！！静まれぇぇええ！！マナーモードはどこぉぉぉぉおお！
+0.382  命が惜しくば、我が目の前から消えろ！！消えろ！！漆黒の悪魔め！！くらえぇぇ！！
+0.382  「私は万能なんだ...だからあなたの為にこれを作ってきた。これは私しか作れないものなんだ。私は選ばれし存在だ！」
+0.376  「世界が凍っていく。我は衣（ころも）を所望するぞ。なんだと？我が衣（ころも）を取らなきゃならないのか？」
+0.371  心がざわめく。この感情は何だ。求めていたもの、握るべきは栄光のはず。なんだ？この感情は...！！！

=== 普通文：下位10（厨二病度が低い） ===
-0.344  マルガレーテ・フォン・ポンメルン（ドイツ語：Margarethe von Pommern, 1366年ごろ - 1407年4月30日）は、オーストリア公エルンストの最初の妃
-0.343  ドバイ博物館（Dubai Museum）はアラブ首長国連邦のドバイにある博物館
-0.342  大槻 文蔵（大槻文藏；おおつき ぶんぞう、1942年（昭和17年）9月25日 - ）は、シテ方観世流の能楽師で、人間国宝である
-0.341  ボグド・ハーン（モンゴル語：ᠪ

## 3. 厨二病ベクトルの評価

In [7]:
print("平均スコア")
print("厨二病:", float(scores_chu.mean()))
print("普通  :", float(scores_nor.mean()))

平均スコア
厨二病: 0.23629070818424225
普通  : -0.14660251140594482
